# Smart Meal Planner AI: Food-101 Classifier Training Notebook

This notebook trains a lightweight MobileNetV2 image classification model on the Food-101 dataset with transfer learning, evaluates on held-out test data, and exports a quantized `.tflite` model for on-device Android inference.

In [ ]:
import os
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models, applications
import matplotlib.pyplot as plt

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 1. Load Dataset
Using TensorFlow Datasets (`tfds.load('food101')`) with official train and validation splits.

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    'food101',
    split=['train', 'validation'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True
)
num_classes = ds_info.features['label'].num_classes
print(f"Number of classes: {num_classes}")

## 2. Preprocessing & Data Augmentation

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, tf.one_hot(label, num_classes)

train_ds = ds_train.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = ds_test.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## 3. Build Transfer Learning Model (MobileNetV2)

In [ ]:
base_model = applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(scale=1.0/127.5, offset=-1.0)(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(256, activation='relu')(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_acc')]
)
model.summary()

## 4. Train & Fine-Tune

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_food_model.keras', save_best_only=True)
]

# Initial training
# history = model.fit(train_ds, validation_data=test_ds, epochs=10, callbacks=callbacks)

## 5. Export to LiteRT / TensorFlow Lite with Float16 Quantization

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open('../models/food_classifier.tflite', 'wb') as f:
    f.write(tflite_model)
print("Model exported successfully!")